In [0]:
from pyspark.sql import functions as F

SOURCE_TABLE = "credit_card_project.silver.cleaned_transactions"
TARGET_TABLE = "credit_card_project.gold.gold_user_behavior"


In [0]:
silver = spark.table(SOURCE_TABLE)

gold_user_behavior = (
    silver
    .groupBy("user_id")
    .agg(
        F.count("*").cast("long").alias("total_transactions"),
        F.sum("amount").cast("decimal(20,2)").alias("total_spend"),
        F.avg("amount").cast("decimal(20,4)").alias("avg_transaction_amount"),
        F.sum(F.when(F.col("is_fraud") == True, 1).otherwise(0)).cast("long").alias("fraud_count"),
        F.countDistinct("card_id").cast("long").alias("unique_cards"),
        F.countDistinct("merchant_id").cast("long").alias("unique_merchants"),
        F.countDistinct("merchant_category_code").cast("long").alias("unique_mccs"),
        F.sum(F.when(F.col("is_online") == True, 1).otherwise(0)).cast("long").alias("_online_count"),
        F.sum(F.when(F.dayofweek("transaction_timestamp").isin(1, 7),1).otherwise(0)).cast("long").alias("_weekend_count"),
        F.sum(F.when(F.hour("transaction_timestamp").between(0, 5),1).otherwise(0)).cast("long").alias("_night_count"))
    .withColumn("fraud_rate",(F.col("fraud_count") / F.col("total_transactions")).cast("double"))
    .withColumn("online_transaction_pct",(F.col("_online_count") * F.lit(100.0) /F.col("total_transactions")).cast("double"))
    .withColumn("weekend_transaction_pct",(F.col("_weekend_count") * F.lit(100.0) /F.col("total_transactions")).cast("double"))
    .withColumn("night_transaction_pct",(F.col("_night_count") * F.lit(100.0) /F.col("total_transactions")).cast("double"))
    .drop("_online_count","_weekend_count","_night_count")
)


In [0]:
gold_user_behavior.printSchema()
display(gold_user_behavior.orderBy("user_id").limit(20))

### Write the Gold table gold_user_behavior

In [0]:
(
    gold_user_behavior.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET_TABLE)
)


In [0]:
display(spark.table(TARGET_TABLE).orderBy("user_id").limit(20))


### Validate Gold grain

expected:
- row_count              = 2,000
- distinct_user_count    = 2,000


In [0]:
gold = spark.table(TARGET_TABLE)

grain_check = gold.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("user_id").alias("distinct_user_count")
)

display(grain_check)


### Validate transaction population

Expected:
- Silver transactions: 24386900
- Gold transactions:   24386900
- Match:               True


In [0]:
silver = spark.table(SOURCE_TABLE)
silver_transaction_count = (
    silver
    .count()
)

gold_transaction_count = (
    gold
    .agg(F.sum("total_transactions"))
    .first()[0]
)

print("Silver transactions:", silver_transaction_count)
print("Gold transactions:  ", gold_transaction_count)
print("Match:              ", silver_transaction_count == gold_transaction_count)


### Validate total spend



In [0]:
silver_total_spend = (
    silver
    .agg(F.sum("amount").alias("total_spend"))
    .first()["total_spend"]
)

gold_total_spend = (
    gold
    .agg(F.sum("total_spend").alias("total_spend"))
    .first()["total_spend"]
)

print("silver.cleaned_transactions total spend:", silver_total_spend)
print("gold.gold_user_behavior total spend:  ", gold_total_spend)
print("Match:             ", silver_total_spend == gold_total_spend)


### Validate fraud

In [0]:
silver_fraud_count = (
    silver
    .filter(F.col("is_fraud") == True)
    .count()
)

gold_fraud_count = (
    gold
    .agg(F.sum("fraud_count").alias("fraud_count"))
    .first()["fraud_count"]
)

print("Silver fraud count:", silver_fraud_count)
print("Gold fraud count:  ", gold_fraud_count)
print("Match:             ", silver_fraud_count == gold_fraud_count)


### Validate percentage ranges

Expected boundaries:
- fraud_rate                 0 → <= 1
- online_transaction_pct     0 → <= 100
- weekend_transaction_pct    0 → <= 100
- night_transaction_pct      0 → <= 100


In [0]:
range_check = gold.select(
    F.min("fraud_rate").alias("min_fraud_rate"),
    F.max("fraud_rate").alias("max_fraud_rate"),

    F.min("online_transaction_pct").alias("min_online_pct"),
    F.max("online_transaction_pct").alias("max_online_pct"),

    F.min("weekend_transaction_pct").alias("min_weekend_pct"),
    F.max("weekend_transaction_pct").alias("max_weekend_pct"),

    F.min("night_transaction_pct").alias("min_night_pct"),
    F.max("night_transaction_pct").alias("max_night_pct")
)

display(range_check)


Validate user population

Expected:
- Users missing from Gold: 0
- Unexpected Gold users:   0



In [0]:
silver_users = (
    silver
    .select("user_id")
    .distinct()
)

gold_users = (
    gold
    .select("user_id")
    .distinct()
)

missing_users = (
    silver_users
    .join(gold_users, "user_id", "left_anti")
    .count()
)

unexpected_users = (
    gold_users
    .join(silver_users, "user_id", "left_anti")
    .count()
)

print("Users missing from Gold:", missing_users)
print("Unexpected Gold users:  ", unexpected_users)


### Final validation summary

In [0]:
assert gold.count() == 2000
assert gold.select("user_id").distinct().count() == 2000

assert gold.agg(F.sum("total_transactions")).first()[0] == 24386900

assert silver_fraud_count == gold_fraud_count
assert silver_total_spend == gold_total_spend

assert missing_users == 0
assert unexpected_users == 0

print("SUCCESS: gold_user_behavior passed all v1 validation checks.")
